# ATS Job Search

In [1]:
import pandas as pd

df = pd.read_csv(
    "../output/UKcompanies_8_sectors_cleaned.csv",
    dtype=str
)

df.shape

(3415689, 25)

In [4]:
import re
import time
import logging
import argparse
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from tqdm import tqdm


In [5]:

GREENHOUSE_JOBS_URL = "https://boards-api.greenhouse.io/v1/boards/{slug}/jobs"
REQUEST_TIMEOUT     = 10          # 秒
SLUG_GUESS_DELAY    = 0.3         # 猜slug时每次请求间隔（秒）
JOBS_FETCH_DELAY    = 1.0         # 拉招聘数据时每次请求间隔（秒）
MAX_SLUG_ATTEMPTS   = 3           # 每家公司最多尝试几个slug候选

In [6]:
# log record
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("greenhouse_pipeline.log", encoding="utf-8")
    ]
)
log = logging.getLogger(__name__)

In [7]:
# slug生成

# 清洗常见后缀
SUFFIXES_TO_STRIP = [
    r'\blimited\b', r'\bltd\b', r'\bplc\b', r'\bllp\b',
    r'\blp\b',      r'\bgroup\b', r'\bholdings\b', r'\buk\b',
    r'\binternational\b', r'\bglobal\b', r'\bservices\b',
    r'\bsolutions\b', r'\btechnologies\b', r'\btech\b',
]

def clean_company_name(name: str) -> str:
    """标准化公司名称：小写、去后缀、去标点"""
    if not isinstance(name, str):
        return ""
    name = name.lower().strip()
    # 去掉 .com / .co.uk / .io 等域名后缀（ASOS.com → asos）
    name = re.sub(r'\.com\b|\.co\.uk\b|\.io\b|\.ai\b', '', name)
    for suffix in SUFFIXES_TO_STRIP:
        name = re.sub(suffix, '', name).strip()
    # 去掉多余空格
    name = re.sub(r'\s+', ' ', name).strip()
    return name


def generate_slug_candidates(company_name: str) -> list[str]:
    """
    给定公司名，生成最多MAX_SLUG_ATTEMPTS个可能的Greenhouse slug候选

    生成逻辑：
      候选1 — 去掉所有非字母数字（最常见的Greenhouse slug格式）
      候选2 — 空格替换为连字符
      候选3 — 只取清洗后名称的第一个词
    """
    cleaned = clean_company_name(company_name)
    if not cleaned:
        return []

    c1 = re.sub(r'[^a-z0-9]', '', cleaned)
    c2 = re.sub(r'[^a-z0-9]', '-', cleaned).strip('-')
    c3 = cleaned.split()[0] if cleaned.split() else ''
    c3 = re.sub(r'[^a-z0-9]', '', c3)

    # 去重、去空、保持顺序
    seen = set()
    candidates = []
    for c in [c1, c2, c3]:
        if c and c not in seen and len(c) >= 2:
            seen.add(c)
            candidates.append(c)

    return candidates[:MAX_SLUG_ATTEMPTS]


In [8]:
# 验证 slug 是否真实存在

def verify_greenhouse_slug(slug: str) -> bool:
    """
    向 Greenhouse Jobs Board API 发请求验证 slug 是否存在
    返回 True 表示该公司确实在 Greenhouse 上有招聘板
    """
    url = GREENHOUSE_JOBS_URL.format(slug=slug)
    try:
        resp = requests.get(url, timeout=REQUEST_TIMEOUT)
        return resp.status_code == 200
    except requests.RequestException:
        return False


def find_slug_for_company(company_number: str, company_name: str) -> dict:
    """
    对一家公司尝试所有slug候选，返回第一个验证成功的结果

    返回：
      {
        'CompanyNumber': ...,
        'CompanyName':   ...,
        'gh_slug':       'monzo' 或 None,
        'slug_source':   'candidate_1' / 'candidate_2' / ... 或 None,
        'gh_found':      True / False,
      }
    """
    candidates = generate_slug_candidates(company_name)
    result = {
        'CompanyNumber': company_number,
        'CompanyName':   company_name,
        'gh_slug':       None,
        'slug_source':   None,
        'gh_found':      False,
    }

    for i, slug in enumerate(candidates, start=1):
        if verify_greenhouse_slug(slug):
            result['gh_slug']     = slug
            result['slug_source'] = f'candidate_{i}'
            result['gh_found']    = True
            log.debug(f"[{company_number}] {company_name} → slug='{slug}' (candidate {i})")
            break
        time.sleep(SLUG_GUESS_DELAY)

    return result



In [9]:
# get job by slug
def fetch_jobs_for_slug(slug: str) -> list[dict]:
    """
    拉取某个 slug 在 Greenhouse 上的所有在招职位
    content=true 会返回职位的完整描述（这里只取 metadata，不取全文）
    """
    url = GREENHOUSE_JOBS_URL.format(slug=slug)
    try:
        resp = requests.get(url, timeout=REQUEST_TIMEOUT)
        if resp.status_code != 200:
            return []
        return resp.json().get('jobs', [])
    except requests.RequestException as e:
        log.warning(f"Failed to fetch jobs for slug '{slug}': {e}")
        return []


def parse_jobs(slug: str, company_number: str, jobs: list[dict]) -> list[dict]:
    """
    将原始 job JSON 解析为扁平的行记录（每个职位一行）
    """
    rows = []
    for job in jobs:
        location_raw = job.get('location', {})
        if isinstance(location_raw, dict):
            location_name = location_raw.get('name', '')
        else:
            location_name = str(location_raw)

        rows.append({
            'CompanyNumber':  company_number,
            'gh_slug':        slug,
            'job_id':         job.get('id'),
            'job_title':      job.get('title', ''),
            'location':       location_name,
            'department':     ', '.join(
                [d.get('name', '') for d in job.get('departments', [])]
            ),
            'updated_at':     job.get('updated_at', ''),
            'absolute_url':   job.get('absolute_url', ''),
        })
    return rows

